In [ ]:
# ===== Colab环境配置 =====
# 运行此cell安装所有依赖（约2-3分钟）
!pip install -q numpy scipy matplotlib pyyaml
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q soundfile librosa
print('环境配置完成！')

# 确认GPU
import torch
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('GPU不可用，请在运行时类型中选择T4 GPU')

# 下载DeepACE代码
import os
if not os.path.exists('DeepACE_torch'):
    print('请上传DeepACE_torch目录到Colab，或从Google Drive挂载')
    print('替代方案：将DeepACE_torch上传为zip文件')

# 生成mini数据集
if not os.path.exists('DeepACE_torch/data'):
    print('运行prepare_mini_dataset.py生成数据集...')
    # 如果有prepare_mini_dataset.py的话
    # !python scripts/prepare_mini_dataset.py
    print('注意：需要先上传代码和数据，或使用合成数据')


# 模块4 第2次课：DeepACE代码解析与运行本notebook将：1. 逐模块解析DeepACE的代码结构2. 追踪完整的数据流（张量形状变化）3. 用mini数据集跑通训练流程4. 加载预训练权重进行推理---

## §1 代码结构总览DeepACE_torch 目录结构：```DeepACE_torch/├── config.yaml      # 训练配置├── model.py         # DeepACE 主模型├── netblocks.py     # 网络子模块（Rectifier, ConvBlock, MaskGenerator等）├── dataset.py       # 数据集加载├── train.py         # 训练入口├── test.py          # 测试入口├── losses.py        # 损失函数├── utils.py         # 工具函数└── data/            # 数据目录（由prepare_mini_dataset.py生成）```> "先见森林，再见树木"——先理解整体架构，再逐模块深入。

In [ ]:
import torchimport yamlimport osimport sys# 添加 DeepACE_torch 到路径DEEPACE_DIR = os.path.join('DeepACE_torch')sys.path.insert(0, DEEPACE_DIR)# 加载配置with open(os.path.join(DEEPACE_DIR, 'config.yaml'), 'r') as f:    config = yaml.safe_load(f)print("=== 配置参数 ===")for key, val in config.items():    if isinstance(val, dict):        print("%s:" % key)        for k, v in val.items():            print("  %s: %s" % (k, v))    else:        print("%s: %s" % (key, val))

## §2 模型实例化与参数统计先创建模型实例，观察参数量。

In [ ]:
from model import DeepACE# 从配置创建模型model_cfg = config['DeepACE']model = DeepACE(    L=model_cfg['L'],    N=model_cfg['N'],    P=model_cfg['P'],    B=model_cfg['B'],    S=model_cfg['S'],    H=model_cfg['H'],    X=model_cfg['X'],    R=model_cfg['R'],    M=model_cfg['M'],    msk_activate=model_cfg['msk_activate'],    causal=model_cfg['causal'])print("DeepACE 模型结构:")print(model)print()# 参数统计total_params = sum(p.numel() for p in model.parameters())trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)print("总参数量: %d (%.2f M)" % (total_params, total_params / 1e6))print("可训练参数量: %d (%.2f M)" % (trainable_params, trainable_params / 1e6))

## §3 数据流追踪：从输入到输出> 这是最重要的部分！跟踪一个假输入经过模型的每一步，观察张量形状变化。

In [ ]:
# 用一个假输入追踪数据流batch_size = 2sr = config['sample_rate']  # 16000duration = config['segment_length']  # 4.0n_samples = int(sr * duration)  # 64000# 模拟输入：带噪语音dummy_input = torch.randn(batch_size, 1, n_samples)print("输入形状:", dummy_input.shape)print("  (batch=%d, channels=1, samples=%d)" % (batch_size, n_samples))print()# 逐步追踪with torch.no_grad():    # 1. Pad signal    padded, rest = model.pad_signal(dummy_input)    print("1. Padding后:", padded.shape, "(rest=%d)" % rest)    # 2. Encoder    encoded = model.encoder(padded)    print("2. Encoder后:", encoded.shape,          "(Conv1d: kernel=%d, stride=%d, in=1, out=%d)" % (              model.enc_kernel_size, model.enc_stride, model.enc_num_feats))    # 3. Rectifier    rectified = model.input_activation(encoded)    print("3. Rectifier后:", rectified.shape)    # 4. Mask Generator    mask = model.mask_generator(rectified)    print("4. Mask后:", mask.shape, "(M=%d个通道)" % model_cfg['M'])    # 5. Apply mask    masked = rectified * mask    print("5. Masked后:", masked.shape)    # 6. Decoder    decoded = model.decoder(masked)    print("6. Decoder后:", decoded.shape,          "(ConvTranspose1d: %d -> %d)" % (model.enc_num_feats, model.out_channels))    # 7. Channel Rebalancer    rebalanced = model.balance(decoded)    print("7. Rebalancer后:", rebalanced.shape)    # 8. Hardtanh    output = model.out_activation(rebalanced)    print("8. Hardtanh后:", output.shape)    # 9. Remove padding    frames_left = model.enc_kernel_size // model.enc_stride    frames_right = -(-(rest + model.enc_stride) // model.enc_stride)    if frames_right > 0:        output = output[..., frames_left:-frames_right]    else:        output = output[..., frames_left:]    print("9. 最终输出:", output.shape)print()print("=" * 50)print("数据流总结:")print("  输入: (%d, 1, %d) -> 输出: %s" % (batch_size, n_samples, output.shape))print("  输出通道数 = %d (对应%d个CI电极)" % (output.shape[1], output.shape[1]))print("  输出帧数 = %d (对应%.1fs音频, stim_rate=%dHz)" % (    output.shape[2], duration, config['stim_rate']))

## §4 核心子模块解析### §4.1 Encoder编码器是一个普通的 Conv1d：- 输入：(batch, 1, T_samples)- 输出：(batch, N, T')- 作用：将原始波形映射到高维特征空间，类似于"学习型时频分析"关键参数：kernel_size=L=32, stride=L/2=16- 类比：stride=16 等价于 hop_length=16，对应 stim_rate=1000Hz

In [ ]:
# Encoder 详细分析encoder = model.encoderprint("Encoder 层详情:")print("  类型: Conv1d")print("  in_channels: %d" % encoder.in_channels)print("  out_channels: %d" % encoder.out_channels)print("  kernel_size: %d" % encoder.kernel_size[0])print("  stride: %d" % encoder.stride[0])print("  padding: %d" % encoder.padding[0])print("  bias: %s" % encoder.bias)print()print("对比传统ACE:")print("  ACE:  分帧(window=128, hop=18) -> FFT(128点) -> 频带功率(22)")print("  DeepACE Encoder: Conv1d(kernel=32, stride=16) -> (N=64个特征)")print()print("Encoder自动学习了类似分帧+频谱分析的操作！")

### §4.2 RectifierRectifier 是一个可学习的激活函数，将编码器的输出分解为正部和负部：```input -> ReLU(input) * w_pos + ReLU(-input) * w_neg```其中 w_pos 和 w_neg 是可学习参数。

In [ ]:
# Rectifier 分析rectifier = model.input_activationprint("Rectifier 详情:")print("  num_features: %d" % rectifier.num_features)# 可学习参数for name, param in rectifier.named_parameters():    print("  参数 %s: shape=%s" % (name, param.shape))# 对比 ReLU vs Rectifiertest_input = torch.linspace(-2, 2, 100).unsqueeze(0).unsqueeze(0)  # (1, 1, 100)import matplotlib.pyplot as pltimport numpy as npfig, ax = plt.subplots(1, 1, figsize=(8, 4))# ReLUrelu_out = torch.relu(test_input).squeeze().numpy()ax.plot(test_input.squeeze().numpy(), relu_out, label="ReLU", linewidth=2)# Rectifier (positive part)w_pos = rectifier.weight_pos.detach().numpy().flatten()[0]w_neg = rectifier.weight_neg.detach().numpy().flatten()[0]x = test_input.squeeze().numpy()rect_out = np.maximum(0, x) * abs(w_pos) + np.maximum(0, -x) * abs(w_neg)ax.plot(x, rect_out, label="Rectifier (w+=%.3f, w-=%.3f)" % (w_pos, w_neg), linewidth=2, linestyle='--')ax.set_title("ReLU vs Rectifier")ax.set_xlabel("Input")ax.set_ylabel("Output")ax.legend()ax.grid(True, alpha=0.3)plt.tight_layout()plt.show()print("Rectifier 比 ReLU 多了一个可学习的负半轴响应！")

### §4.3 Mask GeneratorMask Generator 是 DeepACE 的核心模块，由 R=2 个 Stack 组成，每个 Stack 包含 X=2 个 ConvBlock。每个 ConvBlock 包含：- 1x1 GroupNorm- P- dilated Conv1d- SE (Squeeze-Excitation) Layer- LAuReL (Learnable Augmented Residual Link)- Skip Connection

In [ ]:
# Mask Generator 分析mg = model.mask_generatorprint("Mask Generator 详情:")print("  输入维度: %d" % mg.input_dim)print("  kernel_size: %d" % mg.kernel_size)print("  num_stacks (R): %d" % len(mg.stacks))print()for s_idx, stack in enumerate(mg.stacks):    print("Stack %d:" % s_idx)    print("  ConvBlocks 数量: %d" % len(stack.conv_blocks))    for b_idx, block in enumerate(stack.conv_blocks):        print("  ConvBlock %d:" % b_idx)        # 查看膨胀率        if hasattr(block, 'conv') and hasattr(block.conv, 'dilation'):            print("    dilation: %s" % str(block.conv.dilation))    print()# SE Layer 示例print("SE Layer (Squeeze-Excitation):")print("  作用: 通道注意力机制——让网络学习哪些通道更重要")print("  类似模块3学的注意力，但这里是通道维度的注意力")print()print("LAuReL (Learnable Augmented Residual Link):")print("  作用: 增强的残差连接——不是简单的 x + f(x)，而是 alpha*x + beta*f(x)")print("  alpha, beta 是可学习参数")

### §4.4 Decoder + ChannelRebalancerDecoder 将高维特征映射回22个电极通道。ChannelRebalancer 是CI特有的模块，用于平衡22个通道之间的刺激水平。

In [ ]:
# Decoder 分析decoder = model.decoderprint("Decoder 详情:")print("  类型: %s" % type(decoder).__name__)print("  in_channels: %d (encoder feature dim)" % decoder.in_channels)print("  out_channels: %d (CI电极数)" % decoder.out_channels)print("  kernel_size: %d" % decoder.kernel_size[0])print()# ChannelRebalancer 分析rebalancer = model.balanceprint("ChannelRebalancer 详情:")for name, param in rebalancer.named_parameters():    print("  参数 %s: shape=%s, 值范围=[%.4f, %.4f]" % (        name, param.shape, param.min().item(), param.max().item()))print()print("ChannelRebalancer 对每个电极通道学习一个缩放因子，")print("确保22个通道的刺激水平保持合理的动态范围。")print("这对应了传统ACE中 THR/MCL 的功能，但是由数据驱动的！")

## §5 数据集加载分析 DeepACE 的数据加载方式。

In [ ]:
import math# 理解 block_shift 和帧数计算sr = config['sample_rate']       # 16000stim_rate = config['stim_rate']  # 1000segment_length = config['segment_length']  # 4.0block_shift = int(math.ceil(sr / stim_rate))n_samples = int(sr * segment_length)n_frames = int(math.ceil(n_samples / block_shift))print("=== DeepACE 数据格式 ===")print()print("输入 (mixture WAV):")print("  采样率: %d Hz" % sr)print("  时长: %.1f s" % segment_length)print("  样本数: %d" % n_samples)print()print("目标 (target .mat):")print("  stim_rate: %d Hz" % stim_rate)print("  block_shift: %d samples/frame" % block_shift)print("  总帧数: %d" % n_frames)print("  电极通道: 22")print("  形状: (%d, 22) -> 转置后 -> (22, %d)" % (n_frames, n_frames))print()print("数据集加载流程:")print("  1. 读取 WAV -> (1, 64000)")print("  2. 读取 .mat['lgf'] -> (250, 22) -> 转置 -> (22, 250)")print("  3. 按segment_length分段")print("  4. collate_fn 填充到相同长度后 stack")

In [ ]:
# 检查 mini 数据集是否就绪data_dir = os.path.join(DEEPACE_DIR, 'data')for split in ['train', 'valid', 'test']:    mix_dir = os.path.join(data_dir, split, 'mixture')    if os.path.exists(mix_dir):        wav_files = [f for f in os.listdir(mix_dir) if f.endswith('.wav')]        print("%s mixture: %d 个文件" % (split, len(wav_files)))    else:        print("%s mixture: 目录不存在" % split)    if split != 'test':        tgt_dir = os.path.join(data_dir, split, 'targetLGF')        if os.path.exists(tgt_dir):            mat_files = [f for f in os.listdir(tgt_dir) if f.endswith('.mat')]            print("%s target:  %d 个文件" % (split, len(mat_files)))print()print("如果数据集不存在，请运行:")print("  cd .. && python scripts/prepare_mini_dataset.py")

## §6 运行训练使用 mini 数据集训练 DeepACE（5分钟内可跑完一个epoch）。

In [ ]:
# 检查数据集是否存在，决定训练模式data_dir = os.path.join(DEEPACE_DIR, 'data')train_mix_dir = os.path.join(data_dir, 'train', 'mixture')has_data = os.path.exists(train_mix_dir) and len(os.listdir(train_mix_dir)) > 0if has_data:    print("检测到 mini 数据集，准备训练...")    print()    print("训练参数:")    print("  batch_size: %d" % config['batch_size'])    print("  learning_rate: %s" % config['learning_rate'])    print("  num_epochs: 5 (演示用，完整训练需150)")    print("  loss: %s" % config['loss'])else:    print("未检测到数据集！请先运行 prepare_mini_dataset.py")    print("本notebook将跳过训练步骤，仅演示模型前向传播。")

In [ ]:
if has_data:    import torch.utils.data as data_utils    from dataset import Dataset, collate_fn    # 创建数据集    train_dataset = Dataset(        mixtures_dir=os.path.join(data_dir, 'train', 'mixture'),        sources_dir=os.path.join(data_dir, 'train', 'targetLGF'),        sample_rate=config['sample_rate'],        segment_length=config['segment_length'],        stim_rate=config['stim_rate'],    )    valid_dataset = Dataset(        mixtures_dir=os.path.join(data_dir, 'valid', 'mixture'),        sources_dir=os.path.join(data_dir, 'valid', 'targetLGF'),        sample_rate=config['sample_rate'],        segment_length=config['segment_length'],        stim_rate=config['stim_rate'],    )    train_loader = data_utils.DataLoader(        train_dataset, batch_size=config['batch_size'],        shuffle=True, collate_fn=collate_fn, num_workers=0    )    valid_loader = data_utils.DataLoader(        valid_dataset, batch_size=config['batch_size'],        shuffle=False, collate_fn=collate_fn, num_workers=0    )    print("训练集: %d segments" % len(train_dataset))    print("验证集: %d segments" % len(valid_dataset))    print("训练 batches: %d" % len(train_loader))    print("验证 batches: %d" % len(valid_loader))    # 查看一个batch的数据    mix_batch, tgt_batch = next(iter(train_loader))    print()    print("一个batch的数据:")    print("  mixture: %s" % str(mix_batch.shape))    print("  target:  %s" % str(tgt_batch.shape))else:    print("跳过（数据集不存在）")

In [ ]:
if has_data:    import torch.nn as nn    from torch.optim import Adam    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')    print("使用设备:", device)    model = model.to(device)    optimizer = Adam(model.parameters(), lr=config['learning_rate'])    criterion = nn.MSELoss()    # 训练 5 个 epoch    n_epochs = 5    train_losses = []    valid_losses = []    for epoch in range(n_epochs):        # Training        model.train()        epoch_loss = 0        n_batches = 0        for mix, tgt in train_loader:            mix = mix.to(device)            tgt = tgt.to(device)            optimizer.zero_grad()            output = model(mix)            # 确保输出和目标长度一致            min_len = min(output.shape[2], tgt.shape[2])            output = output[:, :, :min_len]            tgt = tgt[:, :, :min_len]            loss = criterion(output, tgt)            loss.backward()            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)            optimizer.step()            epoch_loss += loss.item()            n_batches += 1        avg_train_loss = epoch_loss / max(n_batches, 1)        train_losses.append(avg_train_loss)        # Validation        model.eval()        val_loss = 0        val_batches = 0        with torch.no_grad():            for mix, tgt in valid_loader:                mix = mix.to(device)                tgt = tgt.to(device)                output = model(mix)                min_len = min(output.shape[2], tgt.shape[2])                output = output[:, :, :min_len]                tgt = tgt[:, :, :min_len]                loss = criterion(output, tgt)                val_loss += loss.item()                val_batches += 1        avg_val_loss = val_loss / max(val_batches, 1)        valid_losses.append(avg_val_loss)        print("Epoch %d/%d: train_loss=%.6f, val_loss=%.6f" % (            epoch + 1, n_epochs, avg_train_loss, avg_val_loss))    # 绘制损失曲线    import matplotlib.pyplot as plt    plt.figure(figsize=(8, 4))    plt.plot(range(1, n_epochs + 1), train_losses, 'o-', label='Train Loss')    plt.plot(range(1, n_epochs + 1), valid_losses, 's-', label='Valid Loss')    plt.xlabel('Epoch')    plt.ylabel('MSE Loss')    plt.title('Training Progress (Mini Dataset)')    plt.legend()    plt.grid(True, alpha=0.3)    plt.tight_layout()    plt.show()else:    print("跳过训练（数据集不存在）")    print("模型未训练，但可以进行前向传播演示...")    # 前向传播演示    device = torch.device('cpu')    model = model.to(device)    dummy = torch.randn(1, 1, 64000).to(device)    with torch.no_grad():        output = model(dummy)    print("前向传播成功！输出形状:", output.shape)    print("输出值范围: [%.6f, %.6f]" % (output.min().item(), output.max().item()))

## §7 推理：加载预训练权重使用预训练模型处理新的语音样本。

In [ ]:
import torchaudioimport scipy.io# 检查预训练权重pretrained_dir = os.path.join('pretrained')pretrained_file = os.path.join(pretrained_dir, 'best_model.pth')if os.path.exists(pretrained_file):    print("找到预训练权重:", pretrained_file)    checkpoint = torch.load(pretrained_file, map_location=device)    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:        model.load_state_dict(checkpoint['model_state_dict'])    else:        model.load_state_dict(checkpoint)    print("预训练权重加载成功！")    HAS_PRETRAINED = Trueelse:    print("未找到预训练权重文件")    print("将使用上一步训练的模型（如果有）或随机初始化的模型")    HAS_PRETRAINED = has_data  # 如果训练过了就用训练后的    if not HAS_PRETRAINED:        print("WARNING: 使用随机初始化的模型，输出不具参考意义")

In [ ]:
# 推理演示import matplotlib.pyplot as pltimport numpy as npmodel.eval()# 使用验证集的一条数据if has_data:    mix, tgt = valid_dataset[0]    mix = mix.unsqueeze(0).to(device)  # (1, 1, T)    with torch.no_grad():        output = model(mix)    # 对齐长度    min_len = min(output.shape[2], tgt.shape[1])    output_np = output[0, :, :min_len].cpu().numpy()  # (22, T')    tgt_np = tgt[:, :min_len].numpy()  # (22, T')    fig, axes = plt.subplots(2, 1, figsize=(14, 6))    im0 = axes[0].imshow(tgt_np, aspect='auto', origin='lower', cmap='hot')    axes[0].set_title("Target (ACE处理干净语音得到的电极图)")    axes[0].set_ylabel("电极通道")    plt.colorbar(im0, ax=axes[0])    im1 = axes[1].imshow(output_np, aspect='auto', origin='lower', cmap='hot')    axes[1].set_title("DeepACE输出 (从带噪语音预测的电极图)")    axes[1].set_ylabel("电极通道")    axes[1].set_xlabel("时间帧")    plt.colorbar(im1, ax=axes[1])    plt.tight_layout()    plt.show()    # 计算误差    error = np.mean((output_np - tgt_np) ** 2)    print("MSE: %.6f" % error)    print("输出形状:", output_np.shape)else:    # 没有数据集时的演示    dummy = torch.randn(1, 1, 64000).to(device)    with torch.no_grad():        output = model(dummy)    output_np = output[0].cpu().numpy()    plt.figure(figsize=(14, 4))    plt.imshow(output_np, aspect='auto', origin='lower', cmap='hot')    plt.title("DeepACE 输出电极图（随机输入，模型未训练）")    plt.ylabel("电极通道")    plt.xlabel("时间帧")    plt.colorbar(label="刺激幅度")    plt.tight_layout()    plt.show()    print("输出形状:", output_np.shape)    print("值范围: [%.6f, %.6f]" % (output_np.min(), output_np.max()))

## §8 通道选择分析：对比ACE与DeepACE观察DeepACE学到的通道选择模式与传统ACE的差异。

In [ ]:
if has_data:    # 分析 DeepACE 的通道选择模式    # 统计每个通道被激活（值>阈值）的频率    threshold = 0.1    n_frames = output_np.shape[1]    channel_activity = np.mean(output_np > threshold, axis=1)  # (22,)    fig, axes = plt.subplots(1, 2, figsize=(12, 4))    # DeepACE 通道激活频率    axes[0].barh(range(22), channel_activity)    axes[0].set_title("DeepACE 通道激活频率")    axes[0].set_xlabel("激活频率")    axes[0].set_ylabel("电极通道")    # Target 通道激活频率    tgt_activity = np.mean(tgt_np > threshold, axis=1)    axes[1].barh(range(22), tgt_activity)    axes[1].set_title("Target 通道激活频率")    axes[1].set_xlabel("激活频率")    axes[1].set_ylabel("电极通道")    plt.tight_layout()    plt.show()    # 对比    print("通道激活频率对比:")    print("通道  DeepACE  Target  差异")    for ch in range(22):        diff = channel_activity[ch] - tgt_activity[ch]        print("%2d    %.3f    %.3f   %+.3f" % (ch+1, channel_activity[ch], tgt_activity[ch], diff))else:    print("需要数据集才能进行通道选择分析")    print()    print("思考题：")    print("1. DeepACE 的通道选择是否和传统ACE的N-of-M策略类似？")    print("2. DeepACE 是否可能学到了一种'自适应N-of-M'——在不同帧选择不同数量的通道？")    print("3. 如果DeepACE的某些通道一直不被激活，这意味着什么？")

## §9 小结本节课我们：1. 从整体到细节解析了DeepACE的代码结构2. 追踪了完整的数据流，理解了每一步的张量形状变化3. 用mini数据集跑通了训练流程4. 进行了推理，对比了DeepACE输出和ACE目标的差异**核心收获**：- DeepACE用编码器-掩码-解码器的模式替代了传统ACE的固定流程- 每个组件都可以在已有知识中找到原型（CNN、注意力、残差连接等）- 数据流中最关键的理解是：输入是带噪语音波形，输出是22通道电极图**课后任务**：- 阅读完整的 train.py 代码，理解训练循环的所有细节- 阅读 losses.py，理解有哪些可用的损失函数- （可选）准备下次课的修改实验方案